# Practical demonstration 2: from super-resolved images to maps

*SSTA workshop, 26 November 2026. Goes with Session 3, "Sample use cases".*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adb-sr/sr-client/blob/main/notebooks/demo2_use_cases.ipynb)

Session 3 ended with a rule: **before you use super-resolution for a task, test it on a small
labelled sample.** In this notebook you run that test for the place you worked on this
morning, and you end with a verdict: would you **deploy** it, **pilot** it, or **not use**
it?

Work in pairs again. **About 25 minutes, six steps.** You do not need to write code:

* **run each grey cell once**, from top to bottom, with **Shift + Enter** (or the ▶ button);
* then use the **menus, buttons and sliders that appear under the cell**. They act at once,
  and your answers are saved as you type. You never need to run a cell twice.

Nothing needs a key or a GPU: the images and the maps were made before the workshop, and the
notebook only downloads them.

| Step | What you do | Time |
|---|---|---|
| 1 | Pick your place and look at the three images | 3 min |
| 2 | **Guess first**: which image will give the best map? | 1 min |
| 3 | See the maps a land-cover model draws from each image | 3 min |
| 4 | Score them against public maps, and check your guess | 4 min |
| 5 | Hunt: find one place where super-resolution rescued the map and one where it failed | 6 min |
| 6 | Use the map for a decision, and give your **verdict** | 8 min |

The four steps of Session 3 are all here:

| Session 3 | In this notebook |
|---|---|
| 1. Label | public maps: Open Buildings (structures) and OpenStreetMap (roads) |
| 2. Train | a land-cover model that was already trained on real high-resolution images |
| 3. Use | we ran it on Sentinel-2, on Stage 1 and on Stage 2 (before the workshop) |
| 4. Score | you compare each map with the public maps |

## 0. Setup

Run the next cell. It holds the helper code; you do not need to read it.

In [ ]:
# @title Helper functions (run this cell; you do not need to read it) { display-mode: "form" }
import json
import os
import re
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
INPUTS = ["Sentinel-2", "Stage 1", "Stage 2"]
KEYS = {"Sentinel-2": "s2", "Stage 1": "stage1", "Stage 2": "stage2"}
COLOURS = {"Sentinel-2": "#a6a6a6", "Stage 1": "#7fbfdf", "Stage 2": "#007DB7"}
TASK_CLASS = {"buildings": 7, "roads": 3}
PIXEL_M = 2.5
OK_COLOUR, FALSE_COLOUR, MISS_COLOUR = (0.10, 0.70, 0.30), (0.20, 0.45, 0.95), (0.90, 0.20, 0.20)


# ---- the data package ---------------------------------------------------------------------

def _fetch(name, url, folder="demo2_data"):
    """A file of the data package: from a local folder, or downloaded once."""
    if Path(url).is_dir():
        return Path(url) / name
    path = Path(folder) / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(f"{url.rstrip('/')}/{name}", path)
    return path


def get_summary(url):
    """Scores of every place (computed before the workshop)."""
    return json.loads(_fetch("summary.json", url).read_text())


class Place:
    """Everything about one place: the three images, the model's map of each, the public
    reference maps and the scores."""

    def __init__(self, name, url):
        slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        z = np.load(_fetch(f"{slug}.npz", url))
        self.name = name
        self.meta = json.loads(str(z["meta"]))
        self.images = {i: z[KEYS[i]].astype("float32") / 10000 for i in INPUTS}   # 4 bands, 0-1
        self.maps = {i: z[f"lc_{KEYS[i]}"] for i in INPUTS}                       # classes 0-7
        self.refs = {"buildings": z["ref_buildings"].astype(bool),
                     "roads": z["ref_roads"].astype(bool)}
        self.valid = z["valid"].astype(bool)
        self.classes = self.meta["classes"]
        self.class_colours = self.meta["class_colours"]
        self.size_m = self.valid.shape[0] * PIXEL_M

    def __repr__(self):
        m = self.meta
        return (f"{self.name}, {m['date']}: {self.size_m / 1000:.0f} km square, "
                f"{m['n_buildings']:,} buildings and {m['n_road_segments']:,} road segments "
                f"in the public maps" +
                (f"; warnings: {', '.join(m['warnings'])}" if m["warnings"] else ""))

    def prediction(self, task, inp):
        """True where the model's map from this input says 'building' (or 'road')."""
        return self.maps[inp] == TASK_CLASS[task]


def _centre(shape, size_m, east_m, north_m):
    """Centre (row, col) and half-width (pixels) of the view, kept inside the area."""
    h, w = shape[-2:]
    half = int(round(size_m / PIXEL_M / 2))
    cy = int(np.clip(h // 2 - north_m / PIXEL_M, half, h - half)) if 2 * half < h else h // 2
    cx = int(np.clip(w // 2 + east_m / PIXEL_M, half, w - half)) if 2 * half < w else w // 2
    return cy, cx, half


def window(arr, size_m=2000, east_m=0, north_m=0):
    """A square of size_m metres cut from an image or map (last two axes are y, x); its
    centre is moved east_m and north_m from the middle of the area."""
    cy, cx, half = _centre(arr.shape, size_m, east_m, north_m)
    return arr[..., max(cy - half, 0):cy + half, max(cx - half, 0):cx + half]


def rgb(img, ceiling=0.3):
    """Natural colour for display from a (4, y, x) reflectance image."""
    return np.clip(np.moveaxis(img[:3], 0, -1) / ceiling, 0, 1)


def _off(axes):
    for ax in np.ravel(axes):
        ax.set_xticks([])
        ax.set_yticks([])


# ---- figures (each returns a matplotlib figure) ------------------------------------------

def fig_locator(place, size_m, east_m, north_m):
    """The whole area, small, with a yellow frame around the current view."""
    cy, cx, half = _centre(place.valid.shape, size_m, east_m, north_m)
    fig, ax = plt.subplots(figsize=(2.4, 2.4))
    ax.imshow(rgb(place.images["Stage 2"][:, ::4, ::4]), interpolation="nearest",
              extent=(0, place.valid.shape[1], place.valid.shape[0], 0))
    ax.add_patch(Rectangle((cx - half, cy - half), 2 * half, 2 * half, fill=False,
                           ec="#ffd400", lw=2))
    ax.set_title("where you are", fontsize=9)
    _off(ax)
    fig.tight_layout()
    return fig


def fig_inputs(place, size_m=1000, east_m=0, north_m=0):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
    for ax, inp in zip(axes, INPUTS):
        ax.imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                  interpolation="nearest")
        ax.set_title(inp + (" (10 m, enlarged)" if inp == "Sentinel-2" else " (2.5 m)"))
    _off(axes)
    fig.suptitle(f"{place.name}: {size_m} m across")
    fig.tight_layout()
    return fig


def fig_landcover(place, size_m=1000, east_m=0, north_m=0):
    """Top: the three images. Bottom: the land-cover map the model drew from each."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10.8))
    cmap = ListedColormap(place.class_colours)
    for j, inp in enumerate(INPUTS):
        axes[0, j].imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                          interpolation="nearest")
        axes[0, j].set_title(inp)
        axes[1, j].imshow(window(place.maps[inp], size_m, east_m, north_m), cmap=cmap, vmin=0,
                          vmax=len(place.classes) - 1, interpolation="nearest")
        axes[1, j].set_title(f"map from {inp}")
    _off(axes)
    fig.legend([Patch(facecolor=c, edgecolor="grey") for c in place.class_colours],
               place.classes, loc="lower center", ncol=8, fontsize=10, frameon=False)
    fig.suptitle(f"{place.name}: {size_m} m across", y=0.995)
    fig.tight_layout(rect=(0, 0.04, 1, 0.98), h_pad=3.0)
    return fig


def fig_reference(place, size_m=2000, east_m=0, north_m=0):
    """The public maps on top of the Stage 2 image: buildings red, roads yellow."""
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(rgb(window(place.images["Stage 2"], size_m, east_m, north_m)),
              interpolation="nearest")
    roads = window(place.refs["roads"], size_m, east_m, north_m)
    buildings = window(place.refs["buildings"], size_m, east_m, north_m)
    over = np.zeros(roads.shape + (4,))
    over[roads] = (1, 0.85, 0, 0.8)
    over[buildings] = (1, 0.2, 0.2, 0.6)
    ax.imshow(over, interpolation="nearest")
    ax.set_title("the public maps: buildings red, roads yellow")
    _off(ax)
    fig.tight_layout()
    return fig


# ---- scoring -----------------------------------------------------------------------------

def scores(pred, ref, valid):
    p, r = pred[valid], ref[valid]
    tp, fp, fn = np.sum(p & r), np.sum(p & ~r), np.sum(~p & r)
    return {"F1": 2 * tp / max(2 * tp + fp + fn, 1), "found": tp / max(tp + fn, 1),
            "correct": tp / max(tp + fp, 1)}


def score_table(place):
    """F1 of each input for buildings and roads, plus the share of the reference that was
    found (recall) and the share of the model's pixels that are correct (precision)."""
    rows = []
    for task in TASK_CLASS:
        for inp in INPUTS:
            s = scores(place.prediction(task, inp), place.refs[task], place.valid)
            rows.append({"task": task, "input": inp, **{k: round(float(v), 2)
                                                        for k, v in s.items()}})
    return pd.DataFrame(rows).set_index(["task", "input"])


def fig_scores(table, title):
    f1 = table["F1"].unstack("input")[INPUTS].loc[list(TASK_CLASS)]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    f1.plot.bar(ax=ax, color=[COLOURS[i] for i in INPUTS], rot=0, width=0.75)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=8)
    ax.set_ylabel("F1 (1 = perfect)")
    ax.set_xlabel("")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, max(0.1, f1.values.max() * 1.25))
    ax.legend(frameon=False, ncol=3, loc="upper left")
    fig.tight_layout()
    return fig


def best_input(table, task, margin=0.02):
    f1 = table["F1"][task]
    best = f1.idxmax()
    if f1[best] - f1["Sentinel-2"] < margin:
        return "no real difference"
    return best


def guess_lines(table, guesses):
    """Was the guess right? One line per task."""
    lines = []
    for task, guess in guesses.items():
        result = best_input(table, task)
        if guess in (None, ""):
            mark = "you did not guess"
        else:
            mark = "as you guessed ✓" if guess == result else f"you guessed: {guess}"
        f1 = table["F1"][task]
        lines.append(f"{task:<10} best: {result:<18} ({mark})   F1 " +
                     ", ".join(f"{i} {f1[i]:.2f}" for i in INPUTS))
    return lines


def _error_image(pred, ref, valid):
    img = np.full(pred.shape + (3,), 0.12)
    img[pred & ref] = OK_COLOUR
    img[pred & ~ref] = FALSE_COLOUR
    img[~pred & ref] = MISS_COLOUR
    img[~valid] = 0
    return img


def fig_errors(place, size_m=500, east_m=0, north_m=0, task="buildings"):
    """Top: the three images. Bottom: the model's map of each against the public map.
    Green: correct. Blue: false alarm (only the model). Red: missed (only the public map)."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10.6))
    valid = window(place.valid, size_m, east_m, north_m)
    ref = window(place.refs[task], size_m, east_m, north_m)
    for j, inp in enumerate(INPUTS):
        axes[0, j].imshow(rgb(window(place.images[inp], size_m, east_m, north_m)),
                          interpolation="nearest")
        axes[0, j].set_title(inp)
        pred = window(place.prediction(task, inp), size_m, east_m, north_m)
        axes[1, j].imshow(_error_image(pred, ref, valid), interpolation="nearest")
        axes[1, j].set_title(f"{task} from {inp}: F1 here {scores(pred, ref, valid)['F1']:.2f}")
    _off(axes)
    fig.legend([Patch(color=c) for c in (OK_COLOUR, FALSE_COLOUR, MISS_COLOUR)],
               ["correct", "false alarm (only the model)", "missed (only the public map)"],
               loc="lower center", ncol=3, frameon=False)
    fig.suptitle(f"{place.name}, {task}: {size_m} m across", y=0.995)
    fig.tight_layout(rect=(0, 0.04, 1, 0.98), h_pad=3.0)
    return fig


# ---- from a map to a decision -----------------------------------------------------------

def per_square(mask, valid, square_m=250):
    """Share of 'yes' pixels in each square of square_m metres (NaN where data is missing)."""
    n = int(square_m / PIXEL_M)
    h, w = (s // n * n for s in mask.shape)
    m = mask[:h, :w].reshape(h // n, n, w // n, n).mean((1, 3))
    v = valid[:h, :w].reshape(h // n, n, w // n, n).mean((1, 3))
    return np.where(v > 0.9, m, np.nan)


def _top(values, k):
    order = np.argsort(-np.nan_to_num(values.ravel(), nan=-1))
    return set(order[:k].tolist())


def decision(place, task, budget=10, square_m=250):
    """You can visit ``budget`` squares. With each input, pick the squares where the model's
    map shows the most buildings (or roads). A pick is a hit if the square is truly among the
    ``budget`` densest according to the public map. Returns (figure, hits per input, text)."""
    truth = per_square(place.refs[task], place.valid, square_m)
    n = int(np.isfinite(truth).sum())
    budget = min(budget, n)
    true_top = _top(truth, budget)
    hits, picks, corr, est = {}, {}, {}, {}
    for inp in INPUTS:
        est[inp] = per_square(place.prediction(task, inp), place.valid, square_m)
        picks[inp] = _top(est[inp], budget)
        hits[inp] = len(picks[inp] & true_top)
        ok = np.isfinite(est[inp]) & np.isfinite(truth)
        corr[inp] = float(np.corrcoef(est[inp][ok], truth[ok])[0, 1])
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.8))
    vmax = np.nanmax(truth)
    axes[0].imshow(truth, cmap="Reds", vmin=0, vmax=vmax)
    axes[0].set_title(f"public map: share of {task}\n(black frames: the {budget} densest)")
    for idx in true_top:
        r, c = divmod(idx, truth.shape[1])
        axes[0].add_patch(Rectangle((c - .5, r - .5), 1, 1, fill=False, ec="k", lw=1.5))
    for ax, inp in zip(axes[1:], INPUTS):
        ax.imshow(est[inp], cmap="Reds", vmin=0, vmax=max(vmax, np.nanmax(est[inp])))
        for idx in picks[inp]:
            r, c = divmod(idx, truth.shape[1])
            ax.add_patch(Rectangle((c - .5, r - .5), 1, 1, fill=False, lw=2,
                                   ec="#1a9641" if idx in true_top else "#d7191c"))
        ax.set_title(f"{inp}: {hits[inp]} of {budget} picks are hits\n"
                     f"(pattern agreement r = {corr[inp]:.2f})")
    _off(axes)
    fig.suptitle(f"{place.name}: {square_m} m squares; green frame = hit, red = wasted visit")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    text = (f"{n} squares of {square_m} m; you can visit {budget}. Picking at random would "
            f"give about {budget * budget / n:.1f} hits.\n" +
            "\n".join(f"  {inp:<11} {hits[inp]:>2} hits of {budget}" for inp in INPUTS))
    return fig, hits, text


# ---- all places --------------------------------------------------------------------------

def all_places(summary, value="F1"):
    """One table for all places: rows = place and task, columns = inputs."""
    rows = []
    for name, s in summary.items():
        for task in TASK_CLASS:
            rows.append({"place": name, "task": task,
                         **{inp: s["scores"][task][inp].get(value) for inp in INPUTS}})
    return pd.DataFrame(rows).set_index(["task", "place"]).sort_index()


def show_all_places(summary, value="F1", label="F1 (1 = perfect)"):
    table = all_places(summary, value)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
    for ax, task in zip(axes, TASK_CLASS):
        t = table.loc[task]
        t.index = [n.split(",")[0].replace(" office", "").replace(" headquarters", " HQ")
                   for n in t.index]
        t[INPUTS].plot.bar(ax=ax, color=[COLOURS[i] for i in INPUTS], rot=0, width=0.78,
                           legend=task == "buildings")
        for c in ax.containers:
            ax.bar_label(c, fmt="%.2f", fontsize=7)
        ax.set_title(task.capitalize())
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", labelsize=8)
    axes[0].legend(frameon=False, ncol=3, loc="upper left")
    fig.tight_layout()
    plt.show()
    return table


PLACES = {'ADB headquarters, Mandaluyong': {'lat': 14.5878,
                                   'lon': 121.0585,
                                   'date': '2026-04-15',
                                   'note': 'dense city: office towers, malls, residential '
                                           'blocks'},
 'DHSUD office, Quezon City': {'lat': 14.6447,
                               'lon': 121.0514,
                               'date': '2026-04-15',
                               'note': 'city: government offices, tree-lined streets, housing'},
 'PhilSA office, Quezon City': {'lat': 14.6096,
                                'lon': 121.0803,
                                'date': '2026-04-15',
                                'note': 'city by the Marikina river: towers, subdivisions, '
                                        'riverside'},
 'Cebu City and Mandaue': {'lat': 10.33,
                           'lon': 123.92,
                           'date': '2026-03-20',
                           'note': 'dense city by the coast: port, warehouses, small houses'},
 'Nueva Ecija rice farms': {'lat': 15.5784,
                            'lon': 120.9726,
                            'date': '2026-03-15',
                            'note': 'rice fields, villages and farm-to-market roads'}}
PLACE_DEFAULT = "Cebu City and Mandaue"
DATA_URL = os.environ.get("DEMO2_DATA_URL", "https://huggingface.co/datasets/adbsr-internal/ssta-hackathon/resolve/main/demo2")


# ---- controls shared by the workshop notebooks: answers that save themselves, figures as images
import html as _html
import io

import ipywidgets as W
from IPython.display import display

ANSWERS = {}    # your answers; saved as soon as you type or click
UI = {}         # the controls of each step (used for testing)
_WIDE = W.Layout(width="520px")
_LABEL = {"description_width": "150px"}


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def _note(text, colour="#333"):
    return f"<div style='color:{colour}; line-height:1.5'>{text}</div>"


def _pre(text):
    return f"<pre style='margin:4px 0; line-height:1.35'>{_html.escape(text)}</pre>"


# ---- answers that save themselves ------------------------------------------------------------

def ask(*questions):
    """Each question is (key, text, kind): kind is "text" or a list of choices. The answer
    is saved in ANSWERS the moment it changes; nothing to run."""
    rows = []
    for key, text, kind in questions:
        saved = W.HTML()
        if kind == "text":
            field = W.Text(value=ANSWERS.get(key, ""), placeholder="type here",
                           continuous_update=True, layout=_WIDE)
        else:
            field = W.ToggleButtons(options=list(kind), value=ANSWERS.get(key),
                                    style={"button_width": "auto"})

        def on_change(change, key=key, saved=saved):
            ANSWERS[key] = change["new"]
            saved.value = _note("saved ✓", "#1a7f37") if change["new"] not in ("", None) else ""
        field.observe(on_change, names="value")
        rows.append(W.VBox([W.HTML(f"<b>{text}</b>"), W.HBox([field, saved])],
                           layout=W.Layout(margin="6px 0 10px 0")))
        rows[-1].field = field
    box = W.VBox(rows, layout=W.Layout(border="1px solid #007DB7", padding="8px 12px",
                                       margin="8px 0"))
    box.fields = [r.field for r in rows]
    display(box)
    return box


# ---- the notebook's controls: menus, buttons and sliders that act at once ---------------------
S = {}          # this session: the place that is loaded, the scores, the decision
REFRESH = []    # what to redraw when the place changes
BEST = ["Sentinel-2", "Stage 1", "Stage 2", "no real difference"]


def _place_needed():
    if "place" not in S:
        display(W.HTML(_note("<b>Run the cell of step 1 first</b>, then run this cell again.",
                             "#b00020")))
        return False
    return True


def _refresh_all():
    for redraw in REFRESH:
        redraw()


# ---- a viewer: move and zoom, the picture follows -------------------------------------------

class Viewer:
    """A figure of the current place with sliders to move the view and buttons to zoom.
    ``render(place, size_m, east_m, north_m, *extra)`` returns the figure; ``extra`` are
    more controls (for example buildings or roads) whose values are passed on."""

    def __init__(self, render, size_m=1000, extra=()):
        self.render, self.extra = render, list(extra)
        self.size = W.ToggleButtons(options=[("250 m", 250), ("500 m", 500), ("1 km", 1000),
                                             ("whole area", 2000)], value=size_m,
                                    description="Zoom", style={"button_width": "90px",
                                                               "description_width": "150px"})
        self.east = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                description="← west | east →", layout=_WIDE, style=_LABEL)
        self.north = W.IntSlider(value=0, min=-750, max=750, step=50, continuous_update=False,
                                 description="← south | north →", layout=_WIDE, style=_LABEL)
        self.locator = W.Image(format="png", layout=W.Layout(width="190px"))
        self.image = W.Image(format="png")
        for w in (self.size, self.east, self.north, *self.extra):
            w.observe(self.draw, names="value")
        controls = W.VBox([*self.extra, self.size, self.east, self.north])
        self.box = W.VBox([W.HBox([controls, self.locator]), self.image])
        REFRESH.append(self.draw)
        self.draw()

    def draw(self, _=None):
        place = S["place"]
        args = (self.size.value, self.east.value, self.north.value)
        self.locator.value = _png(fig_locator(place, *args))
        self.image.value = _png(self.render(place, *args, *[w.value for w in self.extra]))


def _task_buttons(value="buildings"):
    return W.ToggleButtons(options=list(TASK_CLASS), value=value, description="Map of",
                           style={"button_width": "110px", "description_width": "150px"})


# ---- step 1: pick your place ----------------------------------------------------------------

def pick_place(url):
    """A menu of the five places: choosing one loads it, and every step below follows."""
    menu = W.Dropdown(options=list(PLACES), value=PLACE_DEFAULT, description="Place",
                      layout=_WIDE, style=_LABEL)
    status = W.HTML()
    holder = W.VBox([])

    def load(_=None):
        menu.disabled = True
        status.value = _note(f"Loading {_html.escape(menu.value)} ...")
        try:
            S["place"] = Place(menu.value, url)
            S.pop("table", None)
            S.pop("hits", None)
            ANSWERS["PLACE"] = menu.value
            p = S["place"]
            status.value = _note(_html.escape(repr(p)), "#555")
            if holder.children:
                _refresh_all()          # every step below follows the new place
            else:
                holder.children = (Viewer(fig_inputs, size_m=1000).box,)
        except Exception as exc:  # no network, wrong address: say so, do not crash
            status.value = _note(f"Could not load the data: {_html.escape(str(exc))}",
                                 "#b00020")
        finally:
            menu.disabled = False
    menu.observe(load, names="value")
    ui = W.VBox([menu, status, holder])
    ui.menu, ui.status = menu, status
    UI["place"] = ui
    display(ui)
    load()


# ---- step 2: guess --------------------------------------------------------------------------

def step2_guess():
    UI["guess"] = ask(("GUESS_BUILDINGS", "Which image will give the best map of buildings?",
                       BEST),
                      ("GUESS_ROADS", "And the best map of roads?", BEST))


# ---- step 3: the maps -----------------------------------------------------------------------

def step3_maps():
    if not _place_needed():
        return
    viewer = Viewer(fig_landcover, size_m=1000)
    UI["view3"] = viewer
    display(viewer.box)


# ---- step 4: the scores ---------------------------------------------------------------------

def step4_scores():
    """A button that shows the scores (after you have guessed)."""
    if not _place_needed():
        return
    show = W.Button(description="Show the scores", button_style="primary", icon="eye",
                    layout=W.Layout(width="220px"))
    out = W.VBox([])

    def draw(_=None):
        if not show.disabled:
            return                      # not revealed yet
        p = S["place"]
        table = score_table(p)
        S["table"] = table
        guess = {"buildings": ANSWERS.get("GUESS_BUILDINGS"),
                 "roads": ANSWERS.get("GUESS_ROADS")}
        tab = table.copy()
        tab.index = [f"{t}, {i}" for t, i in tab.index]
        out.children = (
            W.HBox([W.Image(value=_png(fig_reference(p)), format="png",
                            layout=W.Layout(width="400px")),
                    W.Image(value=_png(fig_scores(table, f"{p.name}: the land-cover model "
                                                          "against the public maps")),
                            format="png", layout=W.Layout(width="560px"))]),
            W.HTML(_pre("\n".join(guess_lines(table, guess)))),
            W.HTML(tab.to_html(float_format="%.2f")))

    def reveal(_):
        if not (ANSWERS.get("GUESS_BUILDINGS") and ANSWERS.get("GUESS_ROADS")):
            note.value = _note("Make both guesses in step 2 first (or press again to see the "
                               "scores without a guess).", "#b00020")
            if not note.warned:
                note.warned = True
                return
        note.value = ""
        show.disabled = True
        show.description = "Scores shown"
        draw()
    note = W.HTML()
    note.warned = False
    show.on_click(reveal)
    REFRESH.append(draw)
    ui = W.VBox([show, note, out])
    ui.show = show
    UI["scores"] = ui
    display(ui)


# ---- step 5: hunt ---------------------------------------------------------------------------

def step5_hunt():
    if not _place_needed():
        return
    viewer = Viewer(fig_errors, size_m=500, extra=[_task_buttons()])
    UI["view5"] = viewer
    display(viewer.box)
    UI["finds"] = ask(("RESCUE", "A rescue: something Sentinel-2 missed and super-resolution "
                       "found", "text"),
                      ("FAILURE", "A failure: something the super-resolved map gets wrong",
                       "text"))


# ---- step 6: decision and verdict -----------------------------------------------------------

def step6_decision():
    if not _place_needed():
        return
    task = _task_buttons()
    budget = W.IntSlider(value=10, min=3, max=30, step=1, continuous_update=False,
                         description="Squares you can visit", layout=_WIDE, style=_LABEL)
    image = W.Image(format="png")
    text = W.HTML()

    def draw(_=None):
        fig, hits, summary = decision(S["place"], task.value, budget.value)
        S["hits"], S["decision"] = hits, (task.value, budget.value)
        image.value = _png(fig)
        text.value = _pre(summary)
    for w in (task, budget):
        w.observe(draw, names="value")
    REFRESH.append(draw)
    draw()
    ui = W.VBox([task, budget, image, text])
    ui.task, ui.budget = task, budget
    UI["decision"] = ui
    display(ui)


def step6_verdict():
    UI["verdict"] = ask(
        ("VERDICT_FOR", "For which use? (for example: a screening map of built-up areas)",
         "text"),
        ("VERDICT", "Your verdict", ["deploy", "pilot", "do not use"]),
        ("WOULD_CHANGE_MY_MIND", "What would change your mind?", "text"))


# ---- the result card ------------------------------------------------------------------------

def card_text():
    def get(key):
        v = ANSWERS.get(key)
        return "(not answered)" if v in ("", None) else v

    lines = ["=" * 68, f"PLACE     {get('PLACE')}"]
    if "place" in S:
        table = S.get("table")
        if table is None:
            table = score_table(S["place"])
        for t in TASK_CLASS:
            f1 = table["F1"][t]
            lines.append(f"F1        {t:<10} " + ", ".join(f"{i} {f1[i]:.2f}" for i in INPUTS))
    if "hits" in S:
        t, b = S["decision"]
        lines.append(f"DECISION  {t}, {b} squares: hits " +
                     ", ".join(f"{i} {S['hits'][i]}" for i in INPUTS))
    lines += ["-" * 68,
              f"my guess, buildings:     {get('GUESS_BUILDINGS')}",
              f"my guess, roads:         {get('GUESS_ROADS')}",
              f"rescued by SR:           {get('RESCUE')}",
              f"failure:                 {get('FAILURE')}",
              f"VERDICT for {get('VERDICT_FOR')}: {get('VERDICT')}",
              f"what would change it:    {get('WOULD_CHANGE_MY_MIND')}",
              "=" * 68]
    return "\n".join(lines)


def result_card():
    card = W.HTML(_pre(card_text()))
    again = W.Button(description="Update the card", icon="refresh")
    again.on_click(lambda _: setattr(card, "value", _pre(card_text())))
    ui = W.VBox([card, again])
    ui.card, ui.again = card, again
    UI["card"] = ui
    display(ui)

## 1. Pick your place *(3 minutes)*

Take the place you used this morning, so you already know what its images look like. (If you
used your own place this morning, take the listed place you know best.)

| Place | What is there |
|---|---|
| **ADB headquarters, Mandaluyong** | dense city: office towers, malls, residential blocks |
| **DHSUD office, Quezon City** | city: government offices, tree-lined streets, housing |
| **PhilSA office, Quezon City** | city by the Marikina river: towers, subdivisions, riverside |
| **Cebu City and Mandaue** | dense city by the coast: port, warehouses, small houses |
| **Nueva Ecija rice farms** | rice fields, villages and farm-to-market roads |

For each place there are three images of the same 2 km square:

* **Sentinel-2**: the free 10 m image, enlarged to 2.5 m pixels. This is the "do nothing"
  option.
* **Stage 1**: 8 Sentinel-2 images combined. Nothing invented.
* **Stage 2**: the full model, with added detail.

Run the cell and choose your place in the menu (it loads in a few seconds). Move around with
the sliders; the small picture shows where you are. If you change the place later, every
step below follows: there is nothing to run again.

In [ ]:
pick_place(DATA_URL)

## 2. Guess first *(1 minute)*

A land-cover model will draw a map from each of the three images. It was trained on **real
high-resolution images** and has never seen a super-resolved one. We then check its
**buildings** and its **roads** against public maps.

Which image will give the best map? Agree on a guess and click it before you go on.

In [ ]:
step2_guess()

## 3. The maps *(3 minutes)*

The model starts from SatlasPretrain (Allen Institute for AI) and was fine-tuned on
OpenEarthMap: aerial and satellite images from 44 countries, labelled by hand. It gives
every 2.5 m pixel one of 8 classes. It is the model behind the road result in Session 3.

This is the **"reuse"** path from Session 3: nobody labelled anything on super-resolved
images. We only fed the model three different images of your place.

Top row: the three images. Bottom row: the map the model drew from each. Move around and
compare. Where do the maps differ most?

In [ ]:
step3_maps()

## 4. Score against public maps *(4 minutes)*

The reference is two public maps that anyone can download for any place in the Philippines:

* **Open Buildings**: building footprints from Google, Microsoft and OpenStreetMap, combined
  by VIDA. They were made by a model from 50 cm images, so they are good but not perfect.
* **OpenStreetMap roads** (through Overture Maps). Roads are lines, so each one is widened to
  a typical width for its class (12 m for a motorway, 3 m for a farm track).

**F1** measures the overlap between the model's map and the public map, pixel by pixel:
0 = no overlap, 1 = perfect. *found* is the share of the public map's buildings (or roads)
that the model found; *correct* is the share of the model's buildings that are in the public
map.

Press **Show the scores** to see them, and whether your guess was right.

In [ ]:
step4_scores()

The scores look low compared with Session 3. Two reasons: the references are public maps and
not expert labels, and a pixel-by-pixel score punishes every shift of a few metres. Compare
the three inputs with each other, not with 1.

## 5. Hunt: one rescue, one failure *(6 minutes)*

A score hides where the map is right and where it is wrong. Look for yourself. In the bottom
row, **green** is correct, **blue** is a false alarm (only the model says "building") and
**red** is missed (only the public map says so).

Move the view and find:

* a **rescue**: something the map from Sentinel-2 missed and the map from Stage 1 or Stage 2
  found;
* a **failure**: something the super-resolved map gets wrong. Is it a miss, a false alarm, or
  a building in the wrong shape? Could the public map be the one that is wrong?

You know this place: use that. Switch between **buildings** and **roads** with the buttons,
and write your two finds in the blue box (a few words each).

In [ ]:
step5_hunt()

## 6. From a map to a decision *(8 minutes)*

Maps are used to decide something. Here is one such decision, the same kind you will work on
in tomorrow's hackathon:

> Your team can visit only a few 250 m squares of this area (to survey housing, or to check
> road condition). You send the team to the squares where the map shows the **most**
> buildings (or roads). How many of those visits go to the right squares?

A pick counts as a **hit** if the square is truly among the densest according to the public
map. Move the slider (how many squares you can visit) and switch between buildings and
roads, and watch whether super-resolution changes the number of hits. A better F1 does not always change the decision.
With so few squares, one hit more or less can be chance: try several budgets before you
conclude.

In [ ]:
step6_decision()

**Your verdict.** For the use you have in mind at your place, would you:

* **deploy**: use maps from super-resolved images as they are;
* **pilot**: try it on a larger area with proper labels and field checks first;
* **not use** it: plain Sentinel-2 is as good, or the errors are too costly.

"Super-resolution does not help here" is a perfectly good result.

In [ ]:
step6_verdict()

**Your result card.** Run the cell; if you change an answer afterwards, press **Update the
card**. Enter your verdict in the poll (the link is on the slide), and be ready to say in one
sentence why.

In [ ]:
result_card()

---
## For the discussion: all five places

The same scores for every place. Where does super-resolution help most? Where does it make
no difference? Is that what Session 3 would predict?

In [ ]:
summary = get_summary(DATA_URL)
show_all_places(summary).round(2)

**What we found** (our run before the workshop):

* **Buildings**: the score rises with every step, in every place. F1 is 0.15 to 0.26 on
  Sentinel-2, 0.35 to 0.44 on Stage 1 and 0.53 to 0.60 on Stage 2.
* **Roads**: on Sentinel-2 the model finds almost no roads in four of the five places (F1
  0.01 to 0.07). Only around ADB headquarters, where the avenues are wide, does it reach
  0.20. On Stage 2 it reaches 0.20 to 0.44. The largest gain is for the farm roads of Nueva
  Ecija (0.01 to 0.44); the smallest is in Cebu, where the streets between small houses are
  narrower than a 2.5 m pixel can show.
* **The decision changes less than the score.** With 10 squares to visit for buildings, the
  hits were 4, 5 and 5 (Sentinel-2, Stage 1, Stage 2) at ADB headquarters, 4, 4 and 4 at
  DHSUD, 2, 3 and 1 in Cebu, but 3, 4 and 8 in Nueva Ecija. In a dense city almost every
  square is built up, so a sharper map hardly changes which squares come first. Where
  buildings are few and scattered, it does.

This is the pattern of Session 3: a model trained on real high-resolution images cannot read
a 10 m image, and super-resolution lets it work on free Sentinel-2. Whether that changes
what you would *do* depends on the decision.

## If you have time

**Reuse a model, or train your own?** So far we *reused* a model trained on real
high-resolution images. The other path in Session 3 is to *train a model on the images you
will use*. Before the workshop we also trained a quick, simple model on each input (on the
west half of each area, scored on the east half). The table shows its F1.

In [ ]:
all_places(summary, "own_model_F1").round(2)

**The gap disappears.** When a model is trained on each input, the three inputs score about
the same, and plain Sentinel-2 is often slightly ahead (buildings: 0.44 to 0.64 on Sentinel-2
against 0.40 to 0.59 on Stage 2). This quick model only looks at colours and textures around
each pixel, so it is a rough test, but it shows what part of the gain above is: super-resolution
makes Sentinel-2 *look like* the images that existing models were trained on. So:

* if a good model trained on high-resolution images exists for your task, super-resolution
  lets you run it on free Sentinel-2;
* if you train your own model anyway, test whether super-resolution helps before you rely on
  it. Sometimes plain Sentinel-2 is just as good.

**Before you use a map**

* **Check the warnings and the validity map** of every job (this morning's notebook). Near
  water, use Sentinel-2.
* **Spot-check** a sample of the map against a real image or a field visit before acting on
  it.
* **Use maps for areas, not single objects**: screening, planning, trends. Not for decisions
  about one household or one building.
* **Each map shows a period** (the images span 45 days before and after the date), not a
  single day.

**How the maps were made.** The images come from the super-resolution service (the same jobs
as this morning). The land-cover model is public:
[adbsr-internal/satlas-downstream](https://huggingface.co/adbsr-internal/satlas-downstream)
on Hugging Face. The script that made this data package is
[`tools/demos/build_demo2_package.py`](https://github.com/adb-sr/sr-client/blob/main/tools/demos/build_demo2_package.py).

---
Contains modified Copernicus Sentinel data. Building footprints: Google–Microsoft–OSM Open
Buildings by VIDA (ODbL). Roads: © OpenStreetMap contributors (ODbL), via Overture Maps.
Land-cover model: SatlasPretrain (Ai2) fine-tuned on OpenEarthMap; CC BY-NC-SA 4.0,
non-commercial use.